# M05 Character and Control


## Persona vector 演示

我们读取一个小型教学 artifact，观察在轻量 steering 前后，trait score 如何变化。


In [ ]:
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import json
import math
import matplotlib.pyplot as plt

payload = json.loads((root / "artifacts" / "m05_persona_vectors.json").read_text())
traits = ["helpful", "cautious", "concise"]

fig, axes = plt.subplots(1, len(payload["personas"]), figsize=(12, 4), sharey=True)
for ax, persona in zip(axes, payload["personas"]):
    before = [persona["scores_before"][trait] for trait in traits]
    after = [persona["scores_after"][trait] for trait in traits]
    x = range(len(traits))
    ax.bar([index - 0.16 for index in x], before, width=0.32, label="before", color="#999999")
    ax.bar([index + 0.16 for index in x], after, width=0.32, label="after", color="#2c7fb8")
    ax.set_xticks(list(x))
    ax.set_xticklabels(traits, rotation=20)
    ax.set_ylim(0, 1)
    ax.set_title(persona["label_en"])

axes[0].legend()
plt.tight_layout()

def cosine(values_a, values_b):
    numerator = sum(a * b for a, b in zip(values_a, values_b))
    denom = math.sqrt(sum(a * a for a in values_a) * sum(b * b for b in values_b))
    return numerator / denom

reference = payload["personas"][0]
for persona in payload["personas"][1:]:
    score = cosine(reference["vector"], persona["vector"])
    print(f"cosine({reference['label_en']}, {persona['label_en']}) = {score:.3f}")

print("\nSample prompt shifts:")
for prompt in payload["prompts"]:
    print("-", prompt["prompt"])
    print("  before:", prompt["response_before"])
    print("  after :", prompt["response_after"])


## 小结

persona vector 很适合做教学对象，因为它正好站在 interpretability 和 control 的边界上：既能帮助我们理解行为，又不会假装行为问题已经被解决。
